# 178. Rank Scores

## Problem
Scores(id,score): return score and rank, highest score rank 1, ties share ranks, no gaps. Order by score descending.

[Doocs problem reference](https://github.com/doocs/leetcode/blob/main/solution/0100-0199/0178.Rank%20Scores/README_EN.md) · [LeetCode](https://leetcode.com/problems/rank-scores/)

[Community solution index (Silversaya, 2023)](https://leetcode.cn/discuss/post/3573106/zhe-shi-pandas30tian-ji-hua-de-ti-jie-qu-v0wv/) · [LeetCode discussions](https://leetcode.com/problems/rank-scores/solutions/) · [Individual community solution](https://leetcode.cn/problems/rank-scores/solutions/2373534/178-fen-shu-pai-ming-by-yyz999-e9zp/)

Community links are for further reading; the implementations below are independently written.

## Practice first
Run the data cell and try the blank cell before reading solutions.

In [ ]:
import pandas as pd
from pandas.testing import assert_frame_equal
from decimal import Decimal, ROUND_HALF_UP

def percent(n,d):
    return float((Decimal(int(n))*100/Decimal(int(d))).quantize(Decimal('0.01'),rounding=ROUND_HALF_UP)) if d else None

scores = pd.DataFrame([[1,3.5],[2,4],[3,4],[4,3.8]], columns=["id","score"])
expected = pd.DataFrame([[4,1],[4,1],[3.8,2],[3.5,3]], columns=["score","rank"])
display(scores)
display(expected)

In [ ]:
# Your solution here
# result = ...

## Solutions

### Solution 1 — Dense rank
dense 1,1,2；min 1,1,3。

In [ ]:
def solution_1(scores):
    return scores.assign(rank=scores.score.rank(method='dense',ascending=False).astype(int))[['score','rank']].sort_values('score',ascending=False)

result_1 = solution_1(scores.copy())
display(result_1)

### Solution 2 — Mapping with distinct Series
建立分數到順位的對應。

In [ ]:
def solution_2(scores):
    s = scores.score.drop_duplicates().sort_values(ascending=False).reset_index(drop=True)
    r = pd.Series(s.index+1,index=s)
    return scores.assign(rank=scores.score.map(r))[['score','rank']].sort_values('score',ascending=False)

result_2 = solution_2(scores.copy())
display(result_2)

### Solution 3 — Cumulative changes
值改變時才累加順位。

In [ ]:
def solution_3(scores):
    out = scores[['score']].sort_values('score',ascending=False).copy()
    out['rank'] = out.score.ne(out.score.shift()).cumsum()
    return out

result_3 = solution_3(scores.copy())
display(result_3)

## Check all solutions
Rows are normalized only for the example comparison; preserve required ordering in the solutions themselves.

In [ ]:
def normalize(df):
    return df.sort_values(list(df.columns),na_position='last').reset_index(drop=True)
assert_frame_equal(normalize(result_1), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_2), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_3), normalize(expected), check_dtype=False)
print('All example checks passed')

## Review
Compare readability, intermediate data size and handling of duplicate values, missing matches and empty groups.